# 02 — Retrieval Evaluation

Evaluates minsearch, FAISS, and RRF across:
- 2 ground truth files: `gpt-5.4-mini`, `gpt-5.6-luna`
- 2 embedding models: `all-MiniLM-L6-v2`, `multi-qa-MiniLM-L6-cos-v1`

Saves all results to `data/retrieval-eval-results.csv`.

In [1]:
import sys, os, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv(ROOT / '.envrc')

True

In [2]:
def hit_rate_mrr(search_fn, ground_truth, num_results=10):
    hits = 0
    mrr_sum = 0.0
    for _, row in tqdm(ground_truth.iterrows(), total=len(ground_truth), leave=False):
        results = search_fn(row['question'], num_results=num_results)
        result_ids = [str(r['id']) for r in results]
        expected = str(row['movie_id'])
        if expected in result_ids:
            hits += 1
            rank = result_ids.index(expected) + 1
            mrr_sum += 1 / rank
    n = len(ground_truth)
    return round(hits / n, 4), round(mrr_sum / n, 4)

In [3]:
GT_MODELS = ['gpt-5.4-mini', 'gpt-5.6-luna']
EMBEDDING_MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/multi-qa-MiniLM-L6-cos-v1',
]

all_rows = []

for gt_model in GT_MODELS:
    gt = pd.read_csv(ROOT / f'data/ground-truth-retrieval-{gt_model}.csv')
    print(f'\n=== Ground truth: {gt_model} ({len(gt)} pairs) ===')

    for emb_model in EMBEDDING_MODELS:
        print(f'  Loading indexes with {emb_model} ...')
        os.environ['EMBEDDING_MODEL'] = emb_model

        # Reload ingest to rebuild FAISS with the current embedding model
        import movie_assistant.ingest as ingest
        importlib.reload(ingest)

        emb_short = emb_model.split('/')[-1]

        print(f'  minsearch ...')
        hr, mrr = hit_rate_mrr(ingest.search_minsearch, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'minsearch', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

        print(f'  FAISS ({emb_short}) ...')
        hr, mrr = hit_rate_mrr(ingest.search_faiss, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'faiss', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

        print(f'  RRF ...')
        hr, mrr = hit_rate_mrr(ingest.search_rrf, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'rrf', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

results_df = pd.DataFrame(all_rows)
out = ROOT / 'data/retrieval-eval-results.csv'
results_df.to_csv(out, index=False)
print(f'\nSaved to {out}')


=== Ground truth: gpt-5.4-mini (6000 pairs) ===
  Loading indexes with sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

...

    hit_rate=0.3875, mrr=0.1931
  FAISS (multi-qa-MiniLM-L6-cos-v1) ...


  0%|          | 0/6000 [00:00<?, ?it/s]

    hit_rate=0.5557, mrr=0.3569
  RRF ...


  0%|          | 0/6000 [00:00<?, ?it/s]

    hit_rate=0.5815, mrr=0.3605

Saved to /Users/I556249/PycharmProjects/llm-capstone/data/retrieval-eval-results.csv


In [4]:
# Results table sorted by MRR
results_df.sort_values('mrr', ascending=False).reset_index(drop=True)

,gt_model,embedding,method,hit_rate,mrr
0,gpt-5.6-luna,all-MiniLM-L6-v2,faiss,0.5973,0.3856
1,gpt-5.6-luna,all-MiniLM-L6-v2,rrf,0.6092,0.3810
2,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,rrf,0.5815,0.3605
3,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,faiss,0.5557,0.3569
4,gpt-5.4-mini,all-MiniLM-L6-v2,faiss,0.5407,0.3379
5,gpt-5.4-mini,all-MiniLM-L6-v2,rrf,0.5533,0.3326
6,gpt-5.4-mini,multi-qa-MiniLM-L6-cos-v1,faiss,0.5055,0.3149
7,gpt-5.4-mini,multi-qa-MiniLM-L6-cos-v1,rrf,0.5320,0.3114
8,gpt-5.6-luna,all-MiniLM-L6-v2,minsearch,0.3875,0.1931
9,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,minsearch,0.3875,0.1931


## Boost Tuning

Systematically compare different boost weights for minsearch text fields.
Uses `gpt-5.6-luna` ground truth (best GT model) with the default embedding model.

In [7]:
import itertools

os.environ['EMBEDDING_MODEL'] = 'sentence-transformers/all-MiniLM-L6-v2'
import movie_assistant.ingest as ingest
importlib.reload(ingest)

gt_tune = pd.read_csv(ROOT / 'data/ground-truth-retrieval-gpt-5.6-luna.csv')
gt_sample = gt_tune.sample(500, random_state=42).reset_index(drop=True)
print(f'Tuning on {len(gt_sample)} questions')

TITLE_BOOSTS    = [1.0, 2.0, 3.0, 5.0]
KEYWORD_BOOSTS  = [0.5, 1.0, 2.0, 3.0]
OVERVIEW_BOOSTS = [0.5, 1.0, 1.5, 2.0]

total = len(TITLE_BOOSTS) * len(KEYWORD_BOOSTS) * len(OVERVIEW_BOOSTS)
tune_rows = []

for i, (tb, kb, ob) in enumerate(itertools.product(TITLE_BOOSTS, KEYWORD_BOOSTS, OVERVIEW_BOOSTS)):
    boost = {'title': tb, 'keywords': kb, 'overview': ob, 'tagline': 0.5, 'genres': 0.5}
    print(f'[{i+1}/{total}] title={tb}, keywords={kb}, overview={ob}', end=' ... ')
    search_fn = lambda q, b=boost, **_: ingest.search_minsearch(q, boost=b, num_results=10)
    hr, mrr = hit_rate_mrr(search_fn, gt_sample)
    print(f'hit_rate={hr}, mrr={mrr}')
    tune_rows.append({'title': tb, 'keywords': kb, 'overview': ob, 'hit_rate': hr, 'mrr': mrr})

tune_df = pd.DataFrame(tune_rows)
tune_df.to_csv(ROOT / 'data/boost-tuning-results.csv', index=False)
print(f'\nSaved {len(tune_df)} combinations')
tune_df.sort_values('mrr', ascending=False).head(10)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Tuning on 500 questions
[1/64] title=1.0, keywords=0.5, overview=0.5 ... 

  0%|          | 0/500 [00:00<?, ?it/s]

hit_rate=0.324, mrr=0.1723
[2/64] title=1.0, keywords=0.5, overview=1.0 ... 

...

  0%|          | 0/500 [00:00<?, ?it/s]

hit_rate=0.206, mrr=0.1231
[64/64] title=5.0, keywords=3.0, overview=2.0 ... 

  0%|          | 0/500 [00:00<?, ?it/s]

hit_rate=0.246, mrr=0.135

Saved 64 combinations


,title,keywords,overview,hit_rate,mrr
11,1.0,2.0,2.0,0.620,0.4086
7,1.0,1.0,2.0,0.612,0.3980
15,1.0,3.0,2.0,0.600,0.3787
3,1.0,0.5,2.0,0.600,0.3780
10,1.0,2.0,1.5,0.592,0.3597
6,1.0,1.0,1.5,0.592,0.3522
14,1.0,3.0,1.5,0.568,0.3350
2,1.0,0.5,1.5,0.554,0.3245
31,2.0,3.0,2.0,0.526,0.3081
9,1.0,2.0,1.0,0.530,0.3061


In [9]:
best = tune_df.sort_values('mrr', ascending=False).iloc[0]
print('Best boost combination:')
print(f'  title={best.title}, keywords={best.keywords}, overview={best.overview}')
print(f'  hit_rate={best.hit_rate}, mrr={best.mrr}')

# Compare vs default boost
default_boost = {'title': 3.0, 'keywords': 2.0, 'overview': 1.5, 'tagline': 1.0, 'genres': 1.0}
default_fn = lambda q, **_: ingest.search_minsearch(q, boost=default_boost, num_results=10)
default_hr, default_mrr = hit_rate_mrr(default_fn, gt_sample)
print(f'\nDefault boost: hit_rate={default_hr}, mrr={default_mrr}')
print(f'Best boost:    hit_rate={best.hit_rate}, mrr={best.mrr}')


Best boost combination:
  title=1.0, keywords=2.0, overview=2.0
  hit_rate=0.62, mrr=0.4086


  0%|          | 0/500 [00:00<?, ?it/s]


Default boost: hit_rate=0.362, mrr=0.1821
Best boost:    hit_rate=0.62, mrr=0.4086
